<div>
    <img src="../figs/cammp.png" style="height:8em; width:auto; float: right; margin-right: 10px;">
    <img src="../figs/web-design.png" style="height:8em; width:auto; float: right; margin-right: 10px;">
</div>

# Worksheet 3 | Quantum Convolutional Neural Networks

In this worksheet, we explore Quantum Convolutional Neural Networks (QCNNs), which adapt the successful concepts of classical CNNs to the quantum domain. QCNNs are particularly well-suited for image classification and pattern recognition tasks.

**Topics covered:**
- **Convolutional Layers**: Local feature extraction using parameterized quantum gates
- **Pooling Layers**: Dimensionality reduction through quantum measurements
- **Hierarchical Feature Extraction**: Multi-scale processing of information
- **Image Classification**: Applying QCNNs to digit recognition

**Learning Objectives:**
1. Understand the architecture of quantum convolutional neural networks
2. Implement convolutional and pooling layers using PennyLane
3. Train a QCNN for image classification
4. Analyze the expressibility and generalization behavior of QCNNs

## Introduction

Classical CNNs have revolutionized computer vision by leveraging:

1. **Local Connectivity**: Each neuron only connects to a local region of the input
2. **Parameter Sharing**: The same filters are applied across the entire input
3. **Hierarchical Processing**: Stacking layers to extract increasingly abstract features

QCNNs adapt these principles to quantum computing:

1. **Quantum Convolutions**: Parameterized gates acting on neighboring qubits
2. **Quantum Pooling**: Measurement-based dimensionality reduction
3. **Entanglement**: Natural feature extraction through quantum correlations

The QCNN architecture:

```
Input Image → Quantum Encoding → [Conv + Pool] × N → Fully Connected → Output
```

In [27]:
# Setup: Import required libraries
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp
import matplotlib.pyplot as plt
from sklearn import datasets
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Set up quantum device
num_wires = 6
dev = qml.device("default.qubit", wires=num_wires)

print("Setup complete! Ready to build Quantum CNNs.")
print(f"PennyLane version: {qml.__version__}")

Setup complete! Ready to build Quantum CNNs.
PennyLane version: 0.45.0


## 1) Quantum Convolutional Layers

A quantum convolutional layer applies parameterized gates to neighboring qubits, similar to how classical convolutions slide filters across the input:

1. **Single-qubit rotations**: $U_3(\theta, \phi, \lambda)$ gates on each qubit
2. **Two-qubit interactions**: Ising-type gates ($XX, YY, ZZ$) between neighbors

The convolutional operation can be written as:

$$U_{conv}(\theta) = \prod_{(i,j) \in \text{pairs}} \text{Ising}(\theta_{ij}) \cdot \prod_i R_i(\theta_i)$$

where pairs are chosen to cover all neighboring qubit connections.

<div id="exercise" class="alert alert-info">
  <h3><i class="fa fa-laptop" style="font-size:28px"></i> <br><br> a) Implement a Convolutional Layer</h3>
  
  Implement a quantum convolutional layer that:
  
  <ol>
    <li>Applies U3 rotations to each qubit</li>
    <li>Applies IsingZZ gates between neighboring qubits</li>
    <li>Follows a checkerboard pattern for parallel operations</li>
  </ol>
  
  <hr>
  <i class="fas fa-wrench" style="font-size: 20px"></i> &nbsp; <strong>Variables:</strong>
  <ul>
    <li><code>weights</code>: shape (15,) - parameters for U3 and Ising gates</li>
    <li><code>wires</code>: list of qubit indices to apply the layer on</li>
  </ul>
  <hr>
  
  <a href="../help/tipps_03_3a.ipynb"><i class="fas fa-life-ring" style="font-size:20px"></i> &nbsp;Hint</a>
</div>

In [2]:
# Exercise 3a: Implement a Convolutional Layer

def convolutional_layer(weights, wires):
    """
    Apply a quantum convolutional layer.
    
    Args:
        weights: array of shape (15,) with parameters for:
                 - weights[:3]: U3 params for qubit i
                 - weights[3:6]: U3 params for qubit i+1
                 - weights[6:9]: Ising gate params (only weights[8] used for ZZ)
                 - weights[9:12]: U3 params for qubit i (after Ising)
                 - weights[12:]: U3 params for qubit i+1 (after Ising)
        wires: list of wire indices to apply the layer on
    """
    n_wires = len(wires)
    assert n_wires >= 2, "Convolutional layer requires at least 2 qubits"
    
    # YOUR CODE HERE
    # Step 1: Apply U3 rotations to pairs of qubits
    # Step 2: Apply IsingZZ gates between neighbors
    # Step 3: Apply second set of U3 rotations
    pass

# Test the convolutional layer
test_weights = np.random.rand(15)
test_wires = [0, 1, 2, 3]

@qml.qnode(dev)
def test_conv_circuit(weights, wires):
    convolutional_layer(weights, wires)
    return qml.expval(qml.PauliZ(0))

result = test_conv_circuit(test_weights, test_wires)
print(f"Convolutional layer test result: {result:.4f}")
print("\nCircuit structure:")
print(qml.draw(convolutional_layer)(test_weights, test_wires))

## 2) Quantum Pooling Layers

Pooling layers reduce the spatial dimensions of the data. In QCNNs, this is achieved through:

1. **Measurement**: Collapse one qubit and use the result to condition operations on another
2. **Conditional Operations**: Apply gates to the "kept" qubit based on measurement outcomes

The pooling operation can be written as:

$$|\psi\rangle = \sum_i \alpha_i |i\rangle \otimes |j\rangle \rightarrow \sum_i \alpha_i |f(i, j)\rangle$$

where $f$ is a function that combines the states based on measurement outcomes.

<div id="exercise" class="alert alert-info">
  <h3><i class="fa fa-laptop" style="font-size:28px"></i> <br><br> a) Implement a Pooling Layer</h3>
  
  Implement a quantum pooling layer that:
  
  <ol>
    <li>Measures every second qubit (the "pooled" qubits)</li>
    <li>Uses the measurement outcome to condition U3 gates on the neighboring qubit</li>
    <li>Keeps only the unmeasured qubits for the next layer</li>
  </ol>
  
  <hr>
  <i class="fas fa-wrench" style="font-size: 20px"></i> &nbsp; <strong>Variables:</strong>
  <ul>
    <li><code>weights</code>: array of shape (3,) - U3 parameters for conditional gates</li>
    <li><code>wires</code>: list of wire indices to apply pooling on</li>
  </ul>
  <hr>
  
  <a href="../help/tipps_03_3b.ipynb"><i class="fas fa-life-ring" style="font-size:20px"></i> &nbsp;Hint</a>
</div>

In [ ]:
# Exercise 2a: Implement a Pooling Layer

def pooling_layer(weights, wires):
    """
    Apply a quantum pooling layer.
    
    Args:
        weights: array of shape (3,) - U3 parameters for conditional gates
        wires: list of wire indices to apply pooling on
    """
    n_wires = len(wires)
    assert n_wires >= 2, "Pooling layer requires at least 2 qubits"
    
    # YOUR CODE HERE
    # Step 1: For each pair (even index, odd index), measure the odd qubit
    # Step 2: Use the measurement outcome to condition U3 on the even qubit
    # Hint: Use qml.cond() for classical conditioning
    pass

# Test the pooling layer
test_weights = np.random.rand(3)
test_wires = [0, 1, 2, 3]

@qml.qnode(dev)
def test_pool_circuit(weights, wires):
    pooling_layer(weights, wires)
    return qml.probs(wires=[0, 2])

result = test_pool_circuit(test_weights, test_wires)
print(f"Pooling layer test result (probabilities on kept qubits):")
print(result)

## 3) Complete QCNN Architecture

A complete QCNN consists of:

1. **Data Encoding**: Amplitude embedding of the input image
2. **Convolutional Blocks**: Alternating conv and pooling layers
3. **Fully Connected Layer**: Final classification layer
4. **Measurement**: Output class probabilities

The architecture progressively reduces the number of qubits while increasing the expressiveness of the remaining qubits through entanglement and parameterized gates.

<div id="exercise" class="alert alert-info">
  <h3><i class="fa fa-laptop" style="font-size:28px"></i> <br><br> a) Build a Complete QCNN</h3>
  
  Build a complete QCNN for binary classification by:
  
  <ol>
    <li>Using amplitude embedding to encode the input</li>
    <li>Applying 2 convolutional + pooling blocks</li>
    <li>Adding a fully connected layer on the remaining qubits</li>
    <li>Measuring class probabilities</li>
  </ol>
  
  <hr>
  <i class="fas fa-wrench" style="font-size: 20px"></i> &nbsp; <strong>Variables:</strong>
  <ul>
    <li><code>conv_weights</code>: shape (18, 2) - weights for 2 conv layers</li>
    <li><code>fc_weights</code>: shape (4,) - weights for fully connected layer</li>
    <li><code>features</code>: input data to classify</li>
  </ul>
  <hr>
  
  <a href="../help/tipps_03_3c.ipynb"><i class="fas fa-life-ring" style="font-size:20px"></i> &nbsp;Hint</a>
</div>

In [ ]:
# Exercise 3a: Build a Complete QCNN

def conv_and_pooling(kernel_weights, wires):
    """
    Apply both convolutional and pooling layers.
    """
    # Convolution: 15 parameters
    convolutional_layer(kernel_weights[:15], wires)
    # Pooling: 3 parameters
    pooling_layer(kernel_weights[15:], wires)


def dense_layer(weights, wires):
    """
    Apply a fully connected layer using arbitrary unitary.
    """
    qml.ArbitraryUnitary(weights, wires)


@qml.qnode(dev)
def qcnn_classifier(conv_weights, fc_weights, features):
    """
    Complete QCNN for binary classification.
    
    Args:
        conv_weights: shape (18, 2) - weights for 2 conv+pool layers
        fc_weights: shape (4,) - weights for fully connected layer
        features: input data to classify
    
    Returns:
        Class probabilities
    """
    wires = list(range(num_wires))
    
    # YOUR CODE HERE
    # Step 1: Encode features using amplitude embedding
    # Step 2: Apply first conv+pool block
    # Step 3: Keep only half the qubits (every second)
    # Step 4: Apply second conv+pool block
    # Step 5: Apply fully connected layer
    # Step 6: Return probabilities for class 0
    pass

# Test the QCNN
test_conv_weights = np.random.rand(18, 2)
test_fc_weights = np.random.rand(4)
test_features = np.random.rand(2**num_wires)
test_features = test_features / np.linalg.norm(test_features)  # Normalize

result = qcnn_classifier(test_conv_weights, test_fc_weights, test_features)
print(f"QCNN output (probability for class 0): {result[0]:.4f}")
print("\nCircuit structure:")
print(qml.draw(qcnn_classifier)(test_conv_weights, test_fc_weights, test_features))

## 4) Training a QCNN

Training a QCNN follows the same principles as training any variational quantum circuit:

1. **Forward Pass**: Compute predictions using current weights
2. **Loss Computation**: Calculate cross-entropy or MSE
3. **Gradient Computation**: Use the parameter-shift rule
4. **Weight Update**: Apply gradient descent or Adam

For efficiency, we use JAX for automatic differentiation and vectorized operations.

<div id="exercise" class="alert alert-info">
  <h3><i class="fa fa-laptop" style="font-size:28px"></i> <br><br> a) Train the QCNN</h3>
  
  Train the QCNN on the MNIST digit dataset (binary: 0 vs 1):
  
  <ol>
    <li>Load and preprocess the MNIST data</li>
    <li>Initialize the QCNN weights</li>
    <li>Train for a few epochs</li>
    <li>Plot the training curve</li>
  </ol>
  
  <hr>
  <i class="fas fa-wrench" style="font-size: 20px"></i> &nbsp; <strong>Variables:</strong>
  <ul>
    <li><code>n_train</code>: number of training samples</li>
    <li><code>n_epochs</code>: number of training epochs</li>
  </ul>
  <hr>
  
  <a href="../help/tipps_03_3d.ipynb"><i class="fas fa-life-ring" style="font-size:20px"></i> &nbsp;Hint</a>
</div>

In [ ]:
# Exercise 4a: Train the QCNN

# Load MNIST data (digits 0 and 1)
digits = datasets.load_digits()
images, labels = digits.data, digits.target

# Filter for classes 0 and 1
mask = (labels == 0) | (labels == 1)
images = images[mask]
labels = labels[mask]

# Normalize images
images = images / np.linalg.norm(images, axis=1).reshape((-1, 1))

# YOUR CODE HERE
# Step 1: Split data into train and test sets

# Step 2: Initialize weights

# Step 3: Define cost function

# Step 4: Training loop

# Step 5: Plot training curve

print("Training complete! (or implement the training loop)")

<hr>

## Summary

In this worksheet, we explored Quantum Convolutional Neural Networks:

1. **Quantum Convolutional Layers**: Parameterized gates for local feature extraction
2. **Quantum Pooling Layers**: Measurement-based dimensionality reduction
3. **Complete QCNN Architecture**: Combining conv, pooling, and fully connected layers
4. **Training**: Using gradient-based optimization for QCNNs
5. **Generalization Analysis**: Understanding the behavior of QCNNs on small datasets

**Key Takeaways:**
- QCNNs adapt classical CNN concepts to quantum computing
- Convolutional layers extract local features through parameterized gates
- Pooling reduces dimensionality through measurement
- QCNNs can generalize well with limited training data

<hr>

## Discussion Questions

1. How does the quantum pooling differ from classical pooling?
2. What are the advantages of QCNNs over classical CNNs?
3. What challenges might arise when scaling QCNNs to larger images?

<hr>

## Next Steps

Continue to:
- [Worksheet 4: Quantum Kernels](./04_QKM.ipynb) - Kernel methods in quantum space
- [Worksheet 5: Extra Topics](./05_Extra.ipynb) - Advanced topics and real quantum hardware

---

___
<hr>
<img src = "../figs/CC-BY-SA.png" style="height:3em; width:auto"> 
This work is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/">Creative Commons Attribution-ShareAlike 4.0 International License</a>. </br>

Workshop: Coding the QML Future</br>
Authors: Workshop Team